# 📊 NH 차주 데이터 EDA 노트북

> **새 파이프라인 (순수 Pandas 기반)**  
> 이 노트북은 `eda_pipeline/`의 파이프라인을 셀 단위로 실행하고 시각화를 확인합니다.

**실행 순서:** 위에서 아래로 순서대로 실행하세요 (Shift+Enter)

| 단계 | 내용 |
|------|------|
| Step 1 | Raw TXT 로드 & 기초 전처리 |
| Step 2 | 월별 패널 통합 (as-of join) |
| Step 3 | EDA 개별 섹션 시각화 |

## 🔧 환경 설정

In [ ]:
import sys
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# 프로젝트 루트 설정
NOTEBOOK_DIR = Path.cwd()
# 노트북이 eda_pipeline/ 안에 있다면 부모 경로를 루트로
if NOTEBOOK_DIR.name == 'eda_pipeline':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 작업 디렉토리 변경 (상대 경로 일관성)
os.chdir(PROJECT_ROOT)

print(f'✅ Project Root: {PROJECT_ROOT}')
print(f'✅ Input dir: {PROJECT_ROOT / "input"}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 한글 폰트 설정
candidates = ['Malgun Gothic', 'NanumGothic', 'AppleGothic']
available = {f.name for f in fm.fontManager.ttflist}
for font in candidates:
    if font in available:
        plt.rcParams['font.family'] = font
        break
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# 팔레트
PRIMARY   = '#4F6EF5'
SECONDARY = '#F5A623'
DANGER    = '#E84040'
SUCCESS   = '#27AE60'
DARK      = '#1A1D2E'
LIGHT_BG  = '#F0F4FF'

print('✅ 라이브러리 임포트 완료')
print(f'   pandas: {pd.__version__}  numpy: {np.__version__}')

---
## 📁 Step 1: Raw 데이터 로드 & 기초 전처리

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s')

from eda_pipeline.step1_load import RawLoader

loader = RawLoader(data_dir='input')
frames = loader.load_all()

In [ ]:
# 로드 결과 요약
summary = []
for key, df in frames.items():
    summary.append({
        '식별자': key,
        '행수': f'{len(df):,}',
        '컬럼수': len(df.columns),
        '컬럼 목록 (앞 5개)': ', '.join(df.columns[:5].tolist())
    })

pd.DataFrame(summary).set_index('식별자').style.set_table_styles(
    [{'selector': 'th', 'props': [('background-color', PRIMARY), ('color', 'white')]}]
)

In [ ]:
# UPCHE (기업정보) 샘플
print('=== 기업정보 (UPCHE) 상위 5건 ===')
display(frames['upche'].head())

print('\n=== 부도정보 (BUDO) 컬럼 ===')
display(frames['budo'].head())

In [ ]:
# Target 확인
budo = frames['budo']
print(f'전체 차주: {len(budo):,}명')
print(f'부도 발생: {budo["IS_DEFAULT"].sum():,}명')
print(f'부도율: {budo["IS_DEFAULT"].mean()*100:.2f}%')
if 'IS_RECOVERED' in budo.columns:
    print(f'정상화 건수: {budo["IS_RECOVERED"].sum():,}건')

# OBV 월별 분포
if 'obv' in frames:
    obv = frames['obv']
    print(f'\n관찰세부등급 (OBV): {len(obv):,}행, 기간: {obv["BAS_YM"].min()} ~ {obv["BAS_YM"].max()}')

---
## 🔗 Step 2: 월별 패널 통합

In [ ]:
from eda_pipeline.step2_integrate import PanelBuilder

builder = PanelBuilder(frames=frames, output_dir='eda_pipeline/output')
panel = builder.build()

print(f'\n✅ 패널 데이터 생성 완료')
print(f'   Shape: {panel.shape}')
print(f'   컬럼 수: {len(panel.columns)}')

In [ ]:
# 패널 데이터 기본 확인
display(panel.head(3))
print('\n컬럼 목록:')
for i, col in enumerate(panel.columns):
    print(f'  {i+1:3d}. {col}')

In [ ]:
# TRAIN/VALID 분리 확인
train = panel[panel['SPLIT'] == 'TRAIN']
valid = panel[panel['SPLIT'] == 'VALID']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor(LIGHT_BG)

# 행수 비교
axes[0].bar(['TRAIN (2021-23)', 'VALID (2024-)'], 
            [len(train), len(valid)],
            color=[PRIMARY, SECONDARY], edgecolor='white')
for ax, val in zip(axes[0].patches, [len(train), len(valid)]):
    axes[0].text(ax.get_x() + ax.get_width()/2, ax.get_height() + 1000,
                f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('TRAIN/VALID 행수', fontweight='bold')
axes[0].set_facecolor(LIGHT_BG)

# 부도율 비교
if 'IS_BUDO_YN' in panel.columns:
    rates = [train['IS_BUDO_YN'].mean()*100, valid['IS_BUDO_YN'].mean()*100]
    bars = axes[1].bar(['TRAIN (2021-23)', 'VALID (2024-)'], rates,
                       color=[DANGER, SECONDARY], edgecolor='white')
    for bar, rate in zip(bars, rates):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                    f'{rate:.4f}%', ha='center', fontweight='bold')
    axes[1].set_title('TRAIN/VALID 부도율', fontweight='bold')
    axes[1].set_facecolor(LIGHT_BG)

plt.tight_layout()
plt.show()

---
## 📈 Step 3: EDA 개별 분석

### 3-1. 결측치 분석

In [ ]:
# 결측률 계산
miss = (panel.isnull().mean() * 100).sort_values(ascending=False)
miss_nonzero = miss[miss > 0]

print(f'결측치 있는 컬럼: {len(miss_nonzero)}개 / 전체 {len(panel.columns)}개')
print(f'  45%+ (제거 후보): {(miss_nonzero >= 45).sum()}개')
print(f'  20~45% (주의):    {((miss_nonzero >= 20) & (miss_nonzero < 45)).sum()}개')
print(f'  0~20% (양호):     {((miss_nonzero > 0) & (miss_nonzero < 20)).sum()}개')

# 시각화
fig, ax = plt.subplots(figsize=(14, max(6, len(miss_nonzero) * 0.3)))
fig.patch.set_facecolor(LIGHT_BG)
ax.set_facecolor(LIGHT_BG)

colors = [DANGER if v >= 45 else (SECONDARY if v >= 20 else PRIMARY)
          for v in miss_nonzero.values]
ax.barh(miss_nonzero.index[:50], miss_nonzero.values[:50], color=colors[:50])
ax.axvline(45, color=DANGER, linestyle='--', label='45% 임계선')
ax.axvline(20, color=SECONDARY, linestyle='--', label='20% 주의선')
ax.set_xlabel('결측률 (%)')
ax.set_title('컬럼별 결측률 (상위 50개)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 3-2. Target 분포

In [ ]:
if 'IS_BUDO_YN' in panel.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor(LIGHT_BG)
    
    # 파이 차트
    counts = panel['IS_BUDO_YN'].value_counts()
    axes[0].pie(counts, labels=['정상(0)', '부도(1)'],
                colors=[SUCCESS, DANGER], autopct='%1.3f%%',
                startangle=90)
    axes[0].set_title('전체 부도/정상 비율', fontweight='bold')
    axes[0].set_facecolor(LIGHT_BG)
    
    # 월별 부도 건수
    monthly = panel.groupby('BASE_YM')['IS_BUDO_YN'].sum()
    axes[1].bar(range(len(monthly)), monthly.values, color=DANGER, alpha=0.8)
    step = max(1, len(monthly)//12)
    axes[1].set_xticks(range(0, len(monthly), step))
    axes[1].set_xticklabels(monthly.index[::step], rotation=45, ha='right', fontsize=8)
    axes[1].set_title('월별 부도 발생 건수', fontweight='bold')
    axes[1].set_facecolor(LIGHT_BG)
    
    # 업종별 부도율
    if 'STD_INDS_CFC' in panel.columns:
        ind_rate = (panel.groupby('STD_INDS_CFC')['IS_BUDO_YN'].mean() * 100
                    ).sort_values(ascending=False).head(15)
        axes[2].barh(ind_rate.index[::-1], ind_rate.values[::-1], color=PRIMARY)
        axes[2].set_xlabel('부도율 (%)')
        axes[2].set_title('업종별 부도율 (상위 15)', fontweight='bold')
        axes[2].set_facecolor(LIGHT_BG)
    
    plt.tight_layout()
    plt.show()

### 3-3. 주요 피처 분포

In [ ]:
# 주요 수치형 피처 vs Target 박스플롯
key_feats = [c for c in ['OBV_RZVL_POD', 'OBV_LN_BAC', 'CG01_KIS_SCORE',
                          'C302_CRI_ORD', 'AA10_PERS_CNT', 'AC12_TOTAL_KRW_AM']
             if c in panel.columns]

if key_feats and 'IS_BUDO_YN' in panel.columns:
    n = len(key_feats)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    fig.patch.set_facecolor(LIGHT_BG)
    if n == 1: axes = [axes]
    
    for ax, col in zip(axes, key_feats):
        ax.set_facecolor(LIGHT_BG)
        data0 = panel.loc[panel['IS_BUDO_YN'] == 0, col].dropna()
        data1 = panel.loc[panel['IS_BUDO_YN'] == 1, col].dropna()
        
        # 이상값 클리핑 (1%~99%)
        p1, p99 = panel[col].quantile([0.01, 0.99])
        data0 = data0.clip(p1, p99)
        data1 = data1.clip(p1, p99)
        
        bp = ax.boxplot([data0, data1], labels=['정상(0)', '부도(1)'],
                        patch_artist=True)
        bp['boxes'][0].set_facecolor(SUCCESS)
        bp['boxes'][0].set_alpha(0.7)
        bp['boxes'][1].set_facecolor(DANGER)
        bp['boxes'][1].set_alpha(0.7)
        ax.set_title(col[:25], fontsize=9, fontweight='bold')
    
    plt.suptitle('주요 피처별 부도/정상 분포 비교', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 3-4. 시계열 추이

In [ ]:
# 월별 주요 지표 평균 추이
ts_cols = [c for c in ['OBV_RZVL_POD', 'OBV_LN_BAC', 'CG01_KIS_SCORE', 'C302_CRI_ORD']
           if c in panel.columns]

if ts_cols:
    monthly_agg = panel.groupby('BASE_YM')[ts_cols].mean()
    
    n = len(ts_cols)
    fig, axes = plt.subplots(n, 1, figsize=(16, 4*n))
    if n == 1: axes = [axes]
    fig.patch.set_facecolor(LIGHT_BG)
    
    for ax, col in zip(axes, ts_cols):
        ax.set_facecolor(LIGHT_BG)
        x = range(len(monthly_agg))
        ax.plot(x, monthly_agg[col], color=PRIMARY, linewidth=2, label='월평균')
        
        # TRAIN/VALID 구분선
        if '202312' in monthly_agg.index:
            split_idx = monthly_agg.index.tolist().index('202312')
            ax.axvline(split_idx, color=DANGER, linestyle='--', linewidth=1.5,
                       label='TRAIN/VALID 분리 (2023-12)')
            ax.axvspan(0, split_idx, alpha=0.05, color=PRIMARY)
            ax.axvspan(split_idx, len(x), alpha=0.05, color=SECONDARY)
        
        step = max(1, len(monthly_agg)//12)
        ax.set_xticks(range(0, len(monthly_agg), step))
        ax.set_xticklabels(monthly_agg.index[::step], rotation=45, ha='right', fontsize=8)
        ax.set_title(f'월별 {col} 평균 추이', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 3-5. Target과의 상관관계

In [ ]:
if 'IS_BUDO_YN' in panel.columns:
    num_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != 'IS_BUDO_YN' and c != 'ETB_DT']
    
    # 상관 계산
    corrs = panel[num_cols].corrwith(panel['IS_BUDO_YN']).dropna()
    top_pos = corrs.sort_values(ascending=False).head(10)
    top_neg = corrs.sort_values().head(10)
    top_all = pd.concat([top_pos, top_neg]).sort_values(key=abs, ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    fig.patch.set_facecolor(LIGHT_BG)
    ax.set_facecolor(LIGHT_BG)
    
    colors = [DANGER if v > 0 else PRIMARY for v in top_all.values]
    ax.barh(top_all.index[::-1], top_all.values[::-1], color=colors[::-1])
    ax.axvline(0, color=DARK, linewidth=0.8)
    ax.set_title('IS_BUDO_YN과의 상관계수 (양/음 상위 10개)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Pearson r')
    plt.tight_layout()
    plt.show()
    
    print('\n상관계수 상위 10개 (절대값 기준):')
    display(corrs.sort_values(key=abs, ascending=False).head(10).to_frame('correlation').round(4))

### 3-6. 전체 EDA 리포트 자동 생성

In [ ]:
from eda_pipeline.step3_eda import EDAReporter

reporter = EDAReporter(panel=panel, output_dir='eda_pipeline/output')
reporter.run()

print('\n✅ HTML 리포트 생성 완료!')
print('   파일: eda_pipeline/output/eda_report.html')
print('   브라우저에서 열어 확인하세요.')

---
## 📁 생성된 파일 확인

In [ ]:
output_dir = Path('eda_pipeline/output')
print('📁 eda_pipeline/output/ 파일 목록:')
for f in sorted(output_dir.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        size_str = f'{size/1024/1024:.1f} MB' if size > 1024*1024 else f'{size/1024:.1f} KB'
        print(f'  {f.relative_to(output_dir)}  ({size_str})')

In [ ]:
# 패널 데이터 최종 요약
print('=== 최종 패널 데이터 요약 ===')
print(panel.describe(include='all').T[['count', 'unique', 'mean', 'std', 'min', 'max']].to_string())